# Small sequence models on an hourly station series

Next-hour temperature at KHSV (Huntsville) from a 24-hour window of temperature, dewpoint, wind, and sea-level pressure. Three models, in order:

1. Persistence: the last hour is the forecast.
2. A temporal CNN with two dilated 1D convolutions.
3. A 1D Transformer with two encoder layers and a handful of heads.

One year of one station is a few thousand sequences. That fits a free Colab T4 if the window stays at 24 hours and the batch size stays at 64. The point is the training loop and a fair baseline, not a forecast product.

Data are the Iowa Environmental Mesonet ASOS archive. No account.


In [ ]:
# Colab setup.
%pip install -q torch pandas numpy scikit-learn matplotlib


In [ ]:
import numpy as np
import pandas as pd
import requests
from io import StringIO

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

STATION = "HSV"
YEAR = 2024
WINDOW = 24          # hours of history
HORIZON = 1          # predict this many hours ahead
BATCH = 64
EPOCHS = 20
LR = 1e-3
SEED = 7
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", DEVICE)


## Load one year of KHSV

Report type 3 is routine hourly observations. Missing values are marked `M`. A short gap is filled by time interpolation. A gap longer than 3 hours is left missing and the window that touches it is dropped.


In [ ]:
url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
params = {
    "station": STATION,
    "data": ["tmpf", "dwpf", "sknt", "mslp"],
    "year1": YEAR, "month1": 1, "day1": 1,
    "year2": YEAR, "month2": 12, "day2": 31,
    "tz": "Etc/UTC",
    "format": "onlycomma",
    "latlon": "no",
    "elev": "no",
    "missing": "M",
    "trace": "T",
    "direct": "no",
    "report_type": 3,
}
r = requests.get(url, params=params, timeout=120)
r.raise_for_status()
raw = pd.read_csv(StringIO(r.text), na_values=["M"])
raw["valid"] = pd.to_datetime(raw["valid"], utc=True)
raw = raw.set_index("valid").sort_index()

hourly = raw[["tmpf", "dwpf", "sknt", "mslp"]].resample("1h").mean()
gap = hourly.isna().any(axis=1)
# Interpolate only across gaps of 3 hours or less.
hourly = hourly.interpolate(method="time", limit=3)
hourly = hourly.dropna()
print(hourly.shape)
print(hourly.head(3))
print("retained fraction:", len(hourly) / (366 * 24))


## Windows and a time split

Each sample is 24 hours of the four variables. The target is the temperature at the next hour. The last 60 days are the test set, so the split is in time rather than shuffled. Scaling is fit on the training windows only.


In [ ]:
cols = ["tmpf", "dwpf", "sknt", "mslp"]
values = hourly[cols].to_numpy(dtype=np.float32)
times = hourly.index

X, y, t_end = [], [], []
for i in range(WINDOW, len(values) - HORIZON + 1):
    X.append(values[i - WINDOW:i])
    y.append(values[i + HORIZON - 1, 0])  # tmpf is column 0
    t_end.append(times[i + HORIZON - 1])
X = np.stack(X)
y = np.asarray(y, dtype=np.float32)
t_end = pd.DatetimeIndex(t_end)

split = t_end.max() - pd.Timedelta(days=60)
train = t_end < split
test = ~train
print("train", int(train.sum()), "test", int(test.sum()), "split", split)

scaler = StandardScaler()
scaler.fit(X[train].reshape(-1, X.shape[-1]))
def scale(a):
    flat = scaler.transform(a.reshape(-1, a.shape[-1]))
    return flat.reshape(a.shape).astype(np.float32)

X_train, X_test = scale(X[train]), scale(X[test])
y_train, y_test = y[train], y[test]
# Persistence uses the last observed temperature, still in degrees F.
persist = X[test][:, -1, 0]

def loaders():
    tr = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    te = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
    return (
        DataLoader(tr, batch_size=BATCH, shuffle=True),
        DataLoader(te, batch_size=BATCH, shuffle=False),
    )


## Models

The CNN sees the window as channels-last and applies two dilated convolutions, then a linear head. The Transformer uses a learned position vector, two encoder layers, and the last token as the forecast. Both heads emit one number, in degrees F.


In [ ]:
class TemporalCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=2, dilation=2),
            nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=3, padding=4, dilation=4),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(32, 1)

    def forward(self, x):
        # x: (batch, time, features) -> (batch, features, time)
        h = self.net(x.transpose(1, 2)).squeeze(-1)
        return self.head(h).squeeze(-1)


class TinyTransformer(nn.Module):
    def __init__(self, n_features, length, d_model=32, nhead=4, layers=2):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        self.pos = nn.Parameter(torch.zeros(1, length, d_model))
        block = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=64,
            batch_first=True, dropout=0.1,
        )
        self.encoder = nn.TransformerEncoder(block, num_layers=layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.encoder(self.proj(x) + self.pos)
        return self.head(h[:, -1]).squeeze(-1)


def count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

cnn = TemporalCNN(X.shape[-1]).to(DEVICE)
tfm = TinyTransformer(X.shape[-1], WINDOW).to(DEVICE)
print("CNN parameters", count(cnn), "Transformer parameters", count(tfm))


## Train

Mean squared error against the observed next-hour temperature. The test metric reported below is MAE in degrees F, next to persistence.


In [ ]:
def train(model, epochs=EPOCHS):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    tr, _ = loaders()
    history = []
    for epoch in range(epochs):
        total, n = 0.0, 0
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
            n += len(xb)
        history.append(total / n)
    return history


def predict(model):
    model.eval()
    _, te = loaders()
    out = []
    with torch.no_grad():
        for xb, _ in te:
            out.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(out)


def mae(a, b):
    return float(np.mean(np.abs(a - b)))

cnn_hist = train(cnn)
tfm_hist = train(tfm)
cnn_hat = predict(cnn)
tfm_hat = predict(tfm)

print(f"persistence MAE {mae(persist, y_test):.3f} F")
print(f"CNN         MAE {mae(cnn_hat, y_test):.3f} F")
print(f"Transformer MAE {mae(tfm_hat, y_test):.3f} F")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cnn_hist, label="CNN")
axes[0].plot(tfm_hist, label="Transformer")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("train MSE (F^2)")
axes[0].legend()
axes[0].set_title("Training loss")

show = slice(0, 24 * 14)
axes[1].plot(y_test[show], label="observed", color="k", linewidth=1)
axes[1].plot(persist[show], label="persistence", linewidth=1)
axes[1].plot(tfm_hat[show], label="transformer", linewidth=1)
axes[1].set_xlabel("hour in test set")
axes[1].set_ylabel("temperature (F)")
axes[1].legend()
axes[1].set_title("First 14 days of the test set")
fig.tight_layout()
plt.show()


## How to read the result

Persistence is the bar. A next-hour temperature forecast that does not beat the last observation is not using the other variables. A gain of a few tenths of a degree is a real gain on this target. A gain of several degrees usually means a leak, most often scaling fit on the test set or a shuffled split.

Two checks belong next, still on this series:

- Residual by hour of day. If the model only wins at the afternoon peak, it has learned the diurnal cycle, not the weather.
- A second target, dewpoint or wind, with the same windows. Temperature is the easy series.

Do not lengthen the window past 72 hours or raise the batch size past 128 on a free T4. The data, not the GPU, is the limit of a one-station model.
